# 🛰️ FloodShield AI — Model 4A: Satellite Flood Detection (Google Earth Engine + Sentinel-1 SAR)

**Approach:** Change detection — compare pre-flood and during-flood Sentinel-1 SAR imagery to extract flood extent.

**Why Sentinel-1 (radar)?** Optical satellites (Sentinel-2, Landsat) can't see through clouds — but Bangladesh's monsoon skies are often cloudy. **Radar (SAR) penetrates clouds**, making it the standard choice for flood monitoring.

**No training required** — water reflects radar signal weakly (dark pixels), so simple thresholding is enough to detect it.

**Real event used:** August 2024 Eastern Bangladesh Flash Flood (Feni, Cumilla, Noakhali, Lakshmipur, Chattogram, Moulvibazar) — caused by heavy rainfall (19–21 Aug 2024) and a surge of water released from an upstream dam in Tripura, India. 71 deaths, 500,000+ displaced, ~$1.2B in damages.

### How to run:
1. Sign up for a free Google Earth Engine account at [signup.earthengine.google.com](https://signup.earthengine.google.com)
2. Register your Google Cloud project for Earth Engine at `https://console.cloud.google.com/earth-engine/configuration?project=YOUR_PROJECT_ID` (choose "Noncommercial use" → "Community" tier — no billing account required)
3. Run this notebook cell by cell (or Runtime → Run all)


## 1️⃣ Setup: Install + Authenticate Earth Engine

In [1]:
!pip install earthengine-api geemap -q

import ee
import geemap

# First-time auth: a browser link will appear — log in with your Google account and paste the code back
ee.Authenticate(force=True)

# Replace with your own registered GCP project ID
ee.Initialize(project='floodshield-505920')

print("✅ Earth Engine initialized")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 60.7 MB/s eta 0:00:00
✅ Earth Engine initialized


## 2️⃣ Define Area of Interest (District) + Date Range

Using a **real, documented flood event**: the August 2024 Eastern Bangladesh Flash Flood.


In [2]:
# Feni district area (one of the worst-affected districts), approximate bounding box
AOI = ee.Geometry.Rectangle([91.30, 22.85, 91.55, 23.10])  # [minLon, minLat, maxLon, maxLat] -- Feni district

# Pre-flood baseline: the two weeks right before the rainfall began (normal water condition)
PRE_FLOOD_START = '2024-08-01'
PRE_FLOOD_END   = '2024-08-15'

# Flood period: from the cloudburst (19 Aug) through peak flooding
FLOOD_START = '2024-08-21'
FLOOD_END   = '2024-08-30'

print("AOI: Feni district | Pre-flood: Aug 1-15, 2024 | Flood period: Aug 21-30, 2024")
print("This is the 2024 Eastern Bangladesh flash flood (Feni/Cumilla) -- a real, documented event.")
print("You can also change AOI to Cumilla/Noakhali/Lakshmipur/Chattogram/Moulvibazar -- all were affected by the same flood.")


AOI: Feni district | Pre-flood: Aug 1-15, 2024 | Flood period: Aug 21-30, 2024
This is the 2024 Eastern Bangladesh flash flood (Feni/Cumilla) -- a real, documented event.
You can also change AOI to Cumilla/Noakhali/Lakshmipur/Chattogram/Moulvibazar -- all were affected by the same flood.


## 3️⃣ Load Sentinel-1 SAR Collection (VH polarization — best for water detection)

In [3]:
def get_s1_image(start_date, end_date, aoi):
    collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(aoi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.eq('instrumentMode', 'IW'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                  .select('VH'))
    count = collection.size().getInfo()
    print(f"Found {count} images between {start_date} and {end_date}")
    return collection.mosaic().clip(aoi)

pre_flood_img = get_s1_image(PRE_FLOOD_START, PRE_FLOOD_END, AOI)
flood_img = get_s1_image(FLOOD_START, FLOOD_END, AOI)


Found 3 images between 2024-08-01 and 2024-08-15
Found 2 images between 2024-08-21 and 2024-08-30


## 4️⃣ Speckle Filtering (reduce SAR noise)

SAR imagery has natural "speckle noise" — a smoothing filter reduces false detections. Radius increased to 100m (from a typical 50m) to further suppress noise-driven false polygons.


In [4]:
def smooth(image):
    return image.focal_median(radius=100, units='meters')

pre_flood_smooth = smooth(pre_flood_img)
flood_smooth = smooth(flood_img)


## 5️⃣ Water Detection via Thresholding

Low VH backscatter (< threshold dB) indicates water. Typical threshold is **-16 to -18 dB** (standard practice in literature); may need adjustment per region.


In [5]:
WATER_THRESHOLD = -16  # dB, adjust if needed

pre_flood_water = pre_flood_smooth.lt(WATER_THRESHOLD)
flood_water = flood_smooth.lt(WATER_THRESHOLD)

# Flood extent = areas that were dry before but are water now
new_flood_area = flood_water.And(pre_flood_water.Not())

print("Water masks computed.")


Water masks computed.


## 6️⃣ Calculate Flood Area (km²)

In [6]:
area_image = new_flood_area.multiply(ee.Image.pixelArea())
stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=AOI,
    scale=30,
    maxPixels=1e9
)
flood_area_km2 = ee.Number(stats.get('VH')).divide(1e6).getInfo()
print(f"Estimated new flood-affected area: {flood_area_km2:.2f} km²")


Estimated new flood-affected area: 22.33 km²


## 7️⃣ Visualize on Interactive Map

In [7]:
Map = geemap.Map()
Map.centerObject(AOI, 10)

Map.addLayer(pre_flood_smooth, {'min': -25, 'max': 0}, 'Pre-Flood SAR')
Map.addLayer(flood_smooth, {'min': -25, 'max': 0}, 'Flood-Period SAR')
Map.addLayer(new_flood_area.selfMask(), {'palette': ['blue']}, 'New Flood Area')

Map


Map(center=[22.975010461554653, 91.42499999999966], controls=(WidgetControl(options=['position', 'transparent_…

## 8️⃣ Export Flood Extent (GeoJSON — for the dashboard map layer)

⚠️ **Note:** Calling `.getInfo()` directly on a large vector collection can hit Earth Engine's 5,000-element limit (common with noisy SAR results). To avoid this, we:
1. Increase the vectorization `scale` (fewer, larger polygons — less speckle-driven fragmentation)
2. Filter out tiny noise polygons
3. Export asynchronously to **Google Drive** instead of pulling results directly into the notebook


In [8]:
# Vectorize at a coarser scale to reduce speckle-driven polygon fragmentation
vectors = new_flood_area.selfMask().reduceToVectors(
    geometry=AOI,
    scale=50,             # increased from 30 to reduce small noise polygons
    geometryType='polygon',
    maxPixels=1e9,
    eightConnected=True
)

# Drop tiny polygons that are likely noise, not real flood extent
vectors_filtered = vectors.filter(ee.Filter.gt('count', 5))

# Export asynchronously to Google Drive (avoids the getInfo() 5000-element limit)
task = ee.batch.Export.table.toDrive(
    collection=vectors_filtered,
    description='flood_extent_export',
    folder='FloodShieldAI_exports',
    fileFormat='GeoJSON'
)
task.start()

print("✅ Export task started. Check your Google Drive folder 'FloodShieldAI_exports' in 1-5 minutes for the GeoJSON file.")


✅ Export task started. Check your Google Drive folder 'FloodShieldAI_exports' in 1-5 minutes for the GeoJSON file.


## 9️⃣ Check Export Task Status

In [9]:
import time

while task.active():
    print("Still running... (checking again in 30s)")
    time.sleep(30)

print("✅ Done! Task state:", task.status()['state'])


Still running... (checking again in 30s)
✅ Done! Task state: COMPLETED


---
## ✅ What this notebook produces
- Real flood extent detection from Sentinel-1 satellite data (no ML training required)
- Flood area estimate (km²)
- GeoJSON export ready to plug directly into the frontend map (Leaflet/Mapbox)

## ⚠️ Limitations (noted honestly)
- Threshold-based detection is simpler and less accurate than a trained ML segmentation model (see Model 4B)
- Urban areas can produce false positives (buildings sometimes have radar signatures similar to water)
- This is **sufficient for an MVP/demo**, but would need more validation before production use

## Next step
**Model 4B** — train a proper U-Net on the Sen1Floods11 public dataset for more accurate, robust flood segmentation than simple thresholding.
